# 01 - RAG retrieval smoke test

Quick, dependency-light check that the knowledge base loads, chunks, and that
`retrieve_relevant_docs` returns sensible documents for each use case. Works
offline using the keyword fallback retriever (no API key required).

In [ ]:
import sys
from pathlib import Path

# Make the 'src' layout importable when running the notebook from notebooks/.
src = Path.cwd().parent / "src"
if src.exists() and str(src) not in sys.path:
    sys.path.insert(0, str(src))
print("src on path:", str(src))

In [ ]:
from compufix_agents.rag.vectorstore import chunk_documents, load_knowledge_base_documents

docs = load_knowledge_base_documents()
chunks = chunk_documents(docs)
print(f"documents: {len(docs)}  chunks: {len(chunks)}")
for d in docs:
    print(" -", d["source"])

In [ ]:
from compufix_agents.rag.retriever import retrieve_relevant_docs

queries = [
    "ModuleNotFoundError: No module named 'cv2'",
    "Mi internet está muy lento",
    "La computadora está lenta y consume mucha RAM",
]
for q in queries:
    print("=" * 70)
    print("Q:", q)
    for hit in retrieve_relevant_docs(q, k=2):
        print(f"  [{hit['score']}] {hit['source']}")

## End-to-end (analysis only)

Run triage + diagnosis + planning for one input. No actions are executed.

In [ ]:
from compufix_agents.graph.workflow import run_analysis

state = run_analysis("ModuleNotFoundError: No module named 'cv2'")
print("problem type:", state.triage.problem_type.value)
print("entities    :", state.triage.extracted_entities)
print("diagnosis   :", state.diagnosis.diagnosis)
print("plan:")
for s in state.plan.plan:
    print(f"  {s.step}. {s.tool} args={s.args} approval={s.requires_approval}")